In [1]:
%env PGPASSWORD=5J8DhII0RRsPW1
import pandas as pd
import re
from constants.db_connections import ENGINE_READ_ONLY
import os
import paramiko
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from itertools import combinations
from pprint import pprint
from itertools import chain
from sqlalchemy import create_engine
import warnings
mega_qc = pd.read_sql('select * from test_1.mega_table_qc_split', con=ENGINE_READ_ONLY)

env: PGPASSWORD=5J8DhII0RRsPW1


Load double stranded Nova Seq 6 qc data

In [ ]:
ns6_ds_filenames1 = pd.read_csv(r'c:\Users\glj523\OneDrive - University of Copenhagen\data\illumina_analysis\HNLW5DSX5_AOZCK_eDNALib060-Kurt.csv', names=['file_name'])
ns6_ds_filenames2 = pd.read_csv(r'c:\Users\glj523\OneDrive - University of Copenhagen\data\illumina_analysis\HNLW5DSX5_AOZCK_eDNALib060-Thorfinn.csv', names=['file_name'])
ns6_ds_filenames1['Project'] = 'eDNALib060-Kurt'
ns6_ds_filenames2['Project'] = 'eDNALib060-Thorfinn'
ns6_ds_filenames = pd.concat([ns6_ds_filenames1, ns6_ds_filenames2])
ns6_ds_filenames['library_id'] = ns6_ds_filenames['file_name'].apply(lambda x: x.split('-')[0])
ns6_ds_filenames['fastq_id'] = ns6_ds_filenames['file_name'].apply(lambda x: x.split('_')[0])
ns6_ds_multiqc = mega_qc[mega_qc['library_id'].isin(ns6_ds_filenames['library_id'].unique())]
outlier_control = {'library_id': ['LV7009026461', 'LV7009026461'], 'qc_type': ['L004_R1', 'L004_R2'], "fastqc_raw__Total Sequences": [1,1]}
outlier_control = pd.DataFrame(data=outlier_control).reindex(columns=list(ns6_ds_multiqc.columns))
ns6_ds_multiqc = pd.concat([ns6_ds_multiqc, outlier_control]).reset_index(drop=True)
assert set(ns6_ds_filenames["library_id"]) == set(ns6_ds_multiqc["library_id"])

# Merge to get the fastq id and project name in the qc data
ns6_ds_multiqc_merge = ns6_ds_multiqc.merge(ns6_ds_filenames[['Project', 'library_id', 'fastq_id']].drop_duplicates(), 
                                      on='library_id', validate='m:1')

ns6_ds_multiqc_merge['Protocol'] = 'Double'
ns6_ds_multiqc_merge['Platform'] = 'NovaSeq6'

assert len(ns6_ds_multiqc) == len(ns6_ds_multiqc_merge)

ns6_ds_multiqc_merge.to_csv(r'c:\Users\glj523\OneDrive - University of Copenhagen\data\illumina_analysis\ns6_ds_multiqc_all.csv', index=False)

In [106]:
ns6_ss_filenames = pd.read_csv(r'c:\Users\glj523\OneDrive - University of Copenhagen\data\illumina_analysis\BH5F5KDSX7_WBDQ4_new_ssDNALib0019.csv', names=['file_name'])
ns6_ss_filenames = ns6_ss_filenames[~ns6_ss_filenames['file_name'].str.contains('Undetermined')]
ns6_ss_filenames['Project'] = 'ssDNALib0019'

ns6_ss_filenames['library_id'] = ns6_ss_filenames['file_name'].apply(lambda x: x.split('-')[0])
ns6_ss_filenames['fastq_id'] = ns6_ss_filenames['file_name'].apply(lambda x: x.split('_')[0])
ns6_ss_multiqc = mega_qc[mega_qc['library_id'].isin(ns6_ss_filenames['library_id'].unique())]
assert set(ns6_ss_filenames["library_id"]) == set(ns6_ss_multiqc["library_id"])

# Merge to get the fastq id and project name in the qc data
ns6_ss_multiqc_merge = ns6_ss_multiqc.merge(ns6_ss_filenames[['Project', 'library_id', 'fastq_id']].drop_duplicates(), 
                                      on='library_id', validate='m:1')
assert len(ns6_ss_multiqc) == len(ns6_ss_multiqc_merge)
ns6_ss_multiqc_merge['Protocol'] = 'Single'
ns6_ss_multiqc_merge['Platform'] = 'NovaSeq6'

ns6_ss_multiqc_merge.to_csv(r'c:\Users\glj523\OneDrive - University of Copenhagen\data\illumina_analysis\ns6_ss_multiqc_all.csv', index=False)

In [123]:
all_ns6_multiqc = pd.concat([ns6_ss_multiqc_merge, ns6_ds_multiqc_merge]).reset_index(drop=True)
all_ns6_multiqc['Lane'] = all_ns6_multiqc['qc_type'].apply(lambda x: x.split('_')[0])
all_ns6_multiqc['Read Type'] = all_ns6_multiqc['qc_type'].apply(lambda x: x.split('_')[1] if len(x.split('_')) > 1 else None)
all_ns6_multiqc['eDNA Concentration'] = 'Unknown'
all_ns6_multiqc.to_csv(r'c:\Users\glj523\OneDrive - University of Copenhagen\data\illumina_analysis\all_ns6_multiqc.csv', index=False)